# 🏥 Real Dataset Multi-Model Training & Benchmark Suite
### Complete Self-Contained Google Colab Notebook

This notebook trains, evaluates, and benchmarks 4 architectures on your real Chest X-ray dataset:
1. **Baseline ResNet-18** (Classic baseline)
2. **Proposed Attentive-LungNet** (ResNet-18 + CBAM Attention + Dual Head)
3. **Baseline DenseNet-121** (CheXNet medical standard)
4. **Baseline EfficientNet-B0** (Google compound scaled)

---

## Cell 1: Install Dependencies & Detect Hardware (GPU)

In [ ]:
!pip install -q tabulate scikit-learn seaborn matplotlib pandas torchvision torch kagglehub

import os
import time
import copy
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)

# Set seed
torch.manual_seed(42)
np.random.seed(42)

# Hardware Acceleration (CUDA for Colab / MPS for Mac)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"[*] Training Device: {device}")
if device.type == "cuda":
    print(f"[*] GPU Model: {torch.cuda.get_device_name(0)}")
elif device.type == "mps":
    print("[*] Hardware: Apple Silicon GPU (MPS)")

## Cell 2: Load Your Real Dataset
Choose **Option A** (Instant 1-Click Cloud Download in ~30 sec) OR **Option B** (Upload your own `chest_xray.zip`):

In [ ]:
# --- OPTION A: 1-Click Kaggle Auto-Download (Recommended for Colab) ---
try:
    import kagglehub
    print("[*] Downloading Kaggle chest X-ray dataset to Colab cloud storage (~30 seconds)...")
    dl_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
    DATASET_ROOT = os.path.join(dl_path, "chest_xray")
except Exception as e:
    print(f"[!] Fallback to local search: {e}")
    DATASET_ROOT = "/content/dataset"

# Auto-detect train folder location
for root, dirs, _ in os.walk(DATASET_ROOT if os.path.exists(DATASET_ROOT) else "."):
    if "train" in dirs and "test" in dirs:
        DATASET_ROOT = root
        break

# --- OPTION B: Manual Upload Button (if Option A is skipped) ---
if not os.path.exists(os.path.join(DATASET_ROOT, "train")):
    from google.colab import files
    print("👇 Click 'Choose Files' to upload your local chest_xray.zip file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".zip"):
            with zipfile.ZipFile(fn, 'r') as z: z.extractall("/content/dataset")
            for root, dirs, _ in os.walk("/content/dataset"):
                if "train" in dirs: DATASET_ROOT = root; break

print(f"\n[+] Dataset Ready at: {DATASET_ROOT}")
print(f"    Folders found: {os.listdir(DATASET_ROOT)}")

## Cell 3: Define Attention Module (CBAM) & Model Architectures

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        reduced = max(in_planes // ratio, 8)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1 = nn.Conv2d(in_planes, reduced, 1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(reduced, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.fc2(self.relu(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size=7)
    def forward(self, x):
        return x * self.ca(x) * self.sa(x * self.ca(x))

# 1. Baseline ResNet-18
class ResNet18Baseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.net.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.net.fc.in_features, num_classes))
    def forward(self, x):
        return self.net(x)

# 2. Proposed Attentive-LungNet (ResNet-18 + CBAM + Dual Head)
class AttentiveLungNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.features = nn.Sequential(
            base.conv1, base.bn1, base.relu, base.maxpool,
            base.layer1, base.layer2, base.layer3, base.layer4
        )
        self.cbam = CBAM(in_planes=512)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, num_classes))
        self.severity_regressor = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, x, return_severity=False):
        x = self.cbam(self.features(x))
        feat = torch.flatten(self.avgpool(x), 1)
        logits = self.classifier(feat)
        if return_severity:
            return logits, self.severity_regressor(feat) * 100.0
        return logits

# 3. Baseline DenseNet-121
class DenseNet121Baseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.net = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.net.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.net.classifier.in_features, num_classes))
    def forward(self, x):
        return self.net(x)

# 4. Baseline EfficientNet-B0
class EfficientNetB0Baseline(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.net.classifier[1] = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.net.classifier[1].in_features, num_classes))
    def forward(self, x):
        return self.net(x)

## Cell 4: Train All Models Across Epochs & Evaluate on Real Test Set

In [ ]:
# ==============================================================================
# CONFIGURE TRAINING PARAMETERS
# ==============================================================================
EPOCHS = 60          # 60 epochs for complete clinical convergence
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(DATASET_ROOT, "train"), transform=train_tf)
val_dataset   = datasets.ImageFolder(os.path.join(DATASET_ROOT, "val"), transform=eval_tf)
test_dataset  = datasets.ImageFolder(os.path.join(DATASET_ROOT, "test"), transform=eval_tf)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
class_names  = train_dataset.classes

print(f"[*] Real Images Loaded: Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}")
print(f"[*] Classes: {class_names}")

models_dict = {
    "ResNet-18 (Baseline)": ResNet18Baseline().to(device),
    "Attentive-LungNet (Proposed)": AttentiveLungNet().to(device),
    "DenseNet-121 (CheXNet)": DenseNet121Baseline().to(device),
    "EfficientNet-B0 (Baseline)": EfficientNetB0Baseline().to(device)
}

trained_models = {}
histories = {}
test_metrics = {}

# --- Training Loop ---
for name, model in models_dict.items():
    print(f"\n" + "="*70)
    print(f"TRAINING {name} FOR {EPOCHS} EPOCHS")
    print("="*70)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        r_loss, correct, total = 0.0, 0, 0
        for x_b, y_b in train_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out = model(x_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            r_loss += loss.item() * x_b.size(0)
            correct += (torch.argmax(out, dim=1) == y_b).sum().item()
            total += y_b.size(0)
        t_acc = (correct / total) * 100.0
        
        # Validation Phase
        model.eval()
        v_loss, v_corr, v_tot = 0.0, 0, 0
        with torch.no_grad():
            for x_b, y_b in val_loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                out = model(x_b)
                v_loss += criterion(out, y_b).item() * x_b.size(0)
                v_corr += (torch.argmax(out, dim=1) == y_b).sum().item()
                v_tot += y_b.size(0)
        val_acc = (v_corr / v_tot) * 100.0
        scheduler.step()
        
        hist["train_loss"].append(r_loss / total)
        hist["val_loss"].append(v_loss / v_tot)
        hist["train_acc"].append(t_acc)
        hist["val_acc"].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
        print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Acc: {t_acc:.1f}% | Val Acc: {val_acc:.1f}%")
        
    model.load_state_dict(best_wts)
    trained_models[name] = model
    histories[name] = hist
    
    # --- Evaluate on Unseen Real Test Set ---
    model.eval()
    preds, targets, probs = [], [], []
    with torch.no_grad():
        for x_b, y_b in test_loader:
            out = model(x_b.to(device))
            probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            targets.extend(y_b.numpy())
            
    cm = confusion_matrix(targets, preds)
    test_metrics[name] = {
        "Accuracy (%)": round(accuracy_score(targets, preds) * 100.0, 1),
        "Sensitivity/Recall (%)": round(recall_score(targets, preds) * 100.0, 1),
        "Specificity (%)": round((cm[0,0]/(cm[0,0]+cm[0,1]))*100.0, 1),
        "Precision (%)": round(precision_score(targets, preds) * 100.0, 1),
        "F1-Score (%)": round(f1_score(targets, preds) * 100.0, 1),
        "ROC-AUC": round(roc_auc_score(targets, probs), 3),
        "Confusion Matrix": cm
    }

print("\n[+] Training & Evaluation Finished on Real Data!")

## Cell 5: Generate Final Benchmark Comparison Table & Confusion Matrices

In [ ]:
# 1. Display Formatted Benchmark Summary Table
df_summary = pd.DataFrame(test_metrics).T
table_cols = ["Accuracy (%)", "Sensitivity/Recall (%)", "Specificity (%)", "Precision (%)", "F1-Score (%)", "ROC-AUC"]
print("\n" + "="*80)
print("FINAL CLINICAL BENCHMARK TABLE (UNSEEN REAL TEST DATA)")
print("="*80)
display(df_summary[table_cols])

# 2. Plot Side-by-Side Confusion Matrices
fig, axes = plt.subplots(1, len(trained_models), figsize=(5 * len(trained_models), 4.2))
for ax, (name, m) in zip(axes, test_metrics.items()):
    sns.heatmap(m["Confusion Matrix"], annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 13, "weight": "bold"})
    ax.set_title(f"{name}\nAcc: {m['Accuracy (%)']:.1f}% | Rec: {m['Sensitivity/Recall (%)']:.1f}%", fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=10)
    ax.set_ylabel("True Label", fontsize=10)
plt.suptitle("Confusion Matrices on Real Chest X-Ray Test Set", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# 3. Plot Learning Curves Across Epochs
plt.figure(figsize=(10, 4.5))
for name, h in histories.items():
    plt.plot(range(1, EPOCHS + 1), h["val_acc"], label=f"{name} (Val Acc)", lw=2)
plt.title(f"Validation Accuracy Across {EPOCHS} Epochs (Real Chest X-Rays)", fontsize=12, fontweight='bold')
plt.xlabel("Epoch", fontsize=10)
plt.ylabel("Validation Accuracy (%)", fontsize=10)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Cell 6: 📸 Interactive Patient Image Tester
Run this cell and click **'Choose Files'** to upload any single Chest X-ray image file. All 4 models will immediately diagnose it side-by-side!

In [ ]:
from google.colab import files
print("👇 Click 'Choose Files' to upload a patient Chest X-ray image:")
uploaded_patient = files.upload()

for fn in uploaded_patient.keys():
    raw_image = Image.open(fn).convert('RGB')
    inp = eval_tf(raw_image).unsqueeze(0).to(device)
    
    plt.figure(figsize=(4, 4))
    plt.imshow(raw_image)
    plt.title(f"Patient X-Ray: {fn}")
    plt.axis("off")
    plt.show()
    
    print(f"\n{'Model Architecture':<30} | {'Diagnosis':<15} | {'Confidence':<12} | {'Digital Twin Severity'}")
    print("-" * 80)
    
    for name, model in trained_models.items():
        model.eval()
        with torch.no_grad():
            if "Attentive" in name:
                logits, sev = model(inp, return_severity=True)
                sev_text = f"{sev.item():.1f}% aeration loss"
            else:
                logits = model(inp)
                sev_text = "N/A (No Dual Head)"
                
            probs = F.softmax(logits, dim=1)[0]
            pred_idx = torch.argmax(probs).item()
            pred_label = class_names[pred_idx]
            conf = probs[pred_idx].item() * 100.0
            
            print(f"{name:<30} | {pred_label:<15} | {conf:>6.1f}%     | {sev_text}")